In [2]:
#AKT
import heapq
import copy

# Định nghĩa trạng thái đích cho 15-puzzle (0 là ô trống)
GOAL = (
    1, 2, 3, 4,
    5, 6, 7, 8,
    9, 10, 11, 12,
    13, 14, 15, 0
)

class PuzzleNode:
    def __init__(self, board, parent=None, move="", g=0):
        self.board = tuple(board)
        self.parent = parent
        self.move = move
        self.g = g
        self.h = self.calculate_manhattan()
        self.f = self.g + self.h

    def calculate_manhattan(self):
        """Tính khoảng cách Manhattan từ trạng thái hiện tại đến đích"""
        distance = 0
        for i in range(16):
            value = self.board[i]
            if value != 0:
                curr_x, curr_y = divmod(i, 4)
                target_x, target_y = divmod(value - 1, 4)
                distance += abs(curr_x - target_x) + abs(curr_y - target_y)
        return distance

    def get_neighbors(self):
        """Tìm các trạng thái có thể đi tiếp theo"""
        neighbors = []
        zero_idx = self.board.index(0)
        x, y = divmod(zero_idx, 4)

        moves = {
            "Lên": (-1, 0), "Xuống": (1, 0),
            "Trái": (0, -1), "Phải": (0, 1)
        }

        for move_name, (dx, dy) in moves.items():
            nx, ny = x + dx, y + dy
            if 0 <= nx < 4 and 0 <= ny < 4:
                new_idx = nx * 4 + ny
                new_board = list(self.board)
                new_board[zero_idx], new_board[new_idx] = new_board[new_idx], new_board[zero_idx]
                neighbors.append(PuzzleNode(new_board, self, move_name, self.g + 1))
        return neighbors

    def __lt__(self, other):
        return self.f < other.f

def solve_15_puzzle(start_board):
    start_node = PuzzleNode(start_board)
    open_list = [start_node]
    closed_set = set()

    while open_list:
        current_node = heapq.heappop(open_list)

        if current_node.board == GOAL:
            return path_to_root(current_node)

        closed_set.add(current_node.board)

        for neighbor in current_node.get_neighbors():
            if neighbor.board in closed_set:
                continue
            heapq.heappush(open_list, neighbor)

    return None

def path_to_root(node):
    path = []
    while node:
        if node.move:
            path.append((node.move, node.board))
        else:
            path.append(("Bắt đầu", node.board))
        node = node.parent
    return path[::-1]

def print_board(board):
    for i in range(0, 16, 4):
        print(f"{board[i]:2} {board[i+1]:2} {board[i+2]:2} {board[i+3]:2}")
    print("-" * 12)

# --- CHẠY THỬ NGHIỆM ---
# Trạng thái bắt đầu (Xáo trộn nhẹ để đảm bảo tìm ra nghiệm nhanh)
start_state = [
    1, 2, 3, 4,
    5, 6, 7, 8,
    9, 10, 0, 12,
    13, 14, 11, 15
]

print("Trạng thái bắt đầu:")
print_board(start_state)

solution = solve_15_puzzle(start_state)

if solution:
    print(f"Tìm thấy lời giải sau {len(solution)-1} bước!\n")
    for move, board in solution:
        print(f"Bước: {move}")
        print_board(board)
else:
    print("Không tìm thấy lời giải.")

Trạng thái bắt đầu:
 1  2  3  4
 5  6  7  8
 9 10  0 12
13 14 11 15
------------
Tìm thấy lời giải sau 2 bước!

Bước: Bắt đầu
 1  2  3  4
 5  6  7  8
 9 10  0 12
13 14 11 15
------------
Bước: Xuống
 1  2  3  4
 5  6  7  8
 9 10 11 12
13 14  0 15
------------
Bước: Phải
 1  2  3  4
 5  6  7  8
 9 10 11 12
13 14 15  0
------------
